# Session 1: a tiny reaction network and what an experiment can see
**A. Real-world problem.** Burning a fuel is not one step. Molecules turn into other
molecules, which turn into others. A list of such steps, each with a speed (a *rate
constant*), is called a **mechanism**. Real mechanisms have hundreds of steps; we start with
two: A -> B -> C. An experiment cannot watch every molecule, it measures one thing (say, how
much B there is over time). Today you write the mechanism as equations, solve it two ways,
and see what a measurement of B tells you. All numbers are toys.

**Plan.** (1) write the equations, (2) solve them with small time steps, (3) compare with the
exact formula, (4) make a pretend experiment with noise.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the mechanism as equations
A turns into B at speed `k1 * A`; B turns into C at speed `k2 * B`. Here A, B, C are amounts
in mol/L (toy numbers) and k1, k2 are rate constants in 1/s.

    dA/dt = -k1 A          dB/dt = k1 A - k2 B          dC/dt = k2 B

Starting with A = 1, B = 0, C = 0, the exact solution is
`A = exp(-k1 t)` and, for k1 different from k2,
`B = k1 / (k2 - k1) * (exp(-k1 t) - exp(-k2 t))`, and `C = 1 - A - B` because molecules
are not lost.

In [ ]:
def exact(t, k1, k2, a0=1.0):
    """Exact A, B, C at time(s) t (s). Handles the special case k1 == k2."""
    t = np.asarray(t, dtype=float)
    a = a0 * np.exp(-k1 * t)
    if abs(k2 - k1) < 1e-9:
        b = a0 * k1 * t * np.exp(-k1 * t)
    else:
        b = a0 * k1 / (k2 - k1) * (np.exp(-k1 * t) - np.exp(-k2 * t))
    return a, b, a0 - a - b


def peak_time(k1, k2):
    """Time (s) at which B is largest: ln(k2/k1) / (k2 - k1), or 1/k when the two are equal."""
    if abs(k2 - k1) < 1e-9:
        return 1.0 / k1
    return float(np.log(k2 / k1) / (k2 - k1))

## Step 2: solving with small steps (Euler's method)
We do not always have a formula. The idea of Euler's method: over a small time step `dt` the
amounts change by (rate x dt). Repeat. It is easy to write and it gets better as `dt` shrinks.

In [ ]:
def euler(k1, k2, dt, t_end, a0=1.0):
    """Return times and an array of shape (n, 3) with A, B, C, using steps of dt seconds."""
    n = int(round(t_end / dt))
    y = np.zeros((n + 1, 3))
    y[0] = [a0, 0.0, 0.0]
    for i in range(n):
        a, b, _ = y[i]
        y[i + 1] = y[i] + dt * np.array([-k1 * a, k1 * a - k2 * b, k2 * b])
    return np.arange(n + 1) * dt, y

## Step 3: what an experiment measures
A real experiment measures something like the concentration of B at a few times, with some
scatter (noise). We imitate it: take the exact B, and add random noise of 0.01 mol/L.
Then a *calibration* would ask: which k1, k2 make the model pass through these points?

In [ ]:
def pretend_experiment(k1, k2, times, noise=0.01, seed=0):
    """Noisy measurements of B at the given times."""
    rng = np.random.default_rng(seed)
    _, b, _ = exact(times, k1, k2)
    return b + noise * rng.standard_normal(np.size(times))


def misfit(k1, k2, times, measured):
    """Root-mean-square difference between model B and the measurements (mol/L)."""
    _, b, _ = exact(times, k1, k2)
    return float(np.sqrt(np.mean((b - measured) ** 2)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    k1, k2 = 0.5, 1.5
    t = np.linspace(0, 8, 400)
    a, b, c = exact(t, k1, k2)
    print(f"k1 = {k1}, k2 = {k2}: B peaks at t = {peak_time(k1, k2):.4f} s with B = {b.max():.4f} mol/L")
    for dt in (0.5, 0.1, 0.01):
        te, y = euler(k1, k2, dt, 8.0)
        err = np.abs(y[:, 1] - exact(te, k1, k2)[1]).max()
        print(f"Euler dt = {dt:5.2f} s: largest error in B = {err:.4f} mol/L")

    times = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
    meas = pretend_experiment(k1, k2, times)
    print("misfit with the true k1, k2:", round(misfit(k1, k2, times, meas), 4), "mol/L")
    print("misfit with k1 = 1.0, k2 = 0.5:", round(misfit(1.0, 0.5, times, meas), 4), "mol/L")

    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(t, a, label="A")
    ax.plot(t, b, label="B")
    ax.plot(t, c, label="C")
    te, y = euler(k1, k2, 0.5, 8.0)
    ax.plot(te, y[:, 1], "k--", lw=1, label="B by Euler, dt = 0.5 s")
    ax.axvline(peak_time(k1, k2), color="gray", ls=":")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("concentration (mol/L, toy)")
    ax.set_title("A -> B -> C: exact curves and a coarse Euler solution")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "abc_curves.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    ax2.plot(t, b, label="model with true k1, k2")
    ax2.plot(t, exact(t, 1.0, 0.5)[1], "--", label="model with k1 = 1.0, k2 = 0.5")
    ax2.errorbar(times, meas, yerr=0.01, fmt="o", color="k", label="pretend measurements of B")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("concentration of B (mol/L, toy)")
    ax2.set_title("What an experiment sees: B only, with noise")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "experiment_B.png", dpi=120)

## Where this stops
Real mechanisms have hundreds of reactions, with rate constants that depend on temperature.
Session 2 adds that temperature dependence (the Arrhenius form).

## Coding tasks
1. **Easy.** Change `k1, k2` in `lesson.py` to 0.5 and 0.5 (equal). Predict the peak time before you run it, and check that the code handles this special case.
2. **Stretch.** Write `euler_error(dt)` that returns the largest error in B over 8 s, and print it for dt = 0.4, 0.2, 0.1 and 0.05. How does the error change when dt is halved?
3. **Explain.** Explain, using the feed and the drain, why the peak of B comes later when k2 becomes smaller.

## Assessment
- **A. Conceptual.** Why does B rise first and then fall, even though both reactions run all the time? At which moment is its rate of change zero?
- **B. Numerical.** For k1 = 0.4 per s and k2 = 0.8 per s, find the time and height of the peak of B. Check the units and that the answer is sensible.
- **C. Graph and simulation interpretation.** In the concentration graph with k1 = 0.5 and k2 = 1.5, which curve rises fastest at the moment B is at its peak, and why?
- **D. Design and optimisation challenge.** Design: you may measure B at one time only. The unknown mechanism is either (k1, k2) = (0.5, 1.5) or (1.5, 0.5), and the measurement noise is 0.01 mol/L. Which time do you choose, and how well will you tell the two apart?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    t = np.linspace(0, 10, 101)
    a, b, c = L.exact(t, 0.5, 1.5)
    checks.append(("amounts add up to 1 at all times", bool(np.allclose(a + b + c, 1.0))))
    checks.append(("start: A = 1, B = 0, C = 0", a[0] == 1.0 and b[0] == 0.0 and c[0] == 0.0))
    checks.append(("A, B, C never negative", bool(a.min() >= 0 and b.min() >= -1e-12 and c.min() >= -1e-12)))

    # worked example: k1 = 0.5, k2 = 1.5 -> peak at ln(3) / 1 = 1.0986 s, B = (1/3)^1.5 = 0.19245
    tp = L.peak_time(0.5, 1.5)
    bp = float(L.exact(tp, 0.5, 1.5)[1])
    checks.append((f"worked example: B peaks at ln 3 = 1.0986 s (got {tp:.4f}) with B = 0.1925 (got {bp:.4f})",
                   abs(tp - np.log(3)) < 1e-9 and abs(bp - (1 / 3) ** 1.5) < 1e-9))
    # hand calculation at t = 1: B = 0.5 / 1.0 * (exp(-0.5) - exp(-1.5)) = 0.1917
    checks.append(("hand calculation: B(1 s) = 0.5 (e^-0.5 - e^-1.5) = 0.1917",
                   abs(float(L.exact(1.0, 0.5, 1.5)[1]) - 0.5 * (np.exp(-0.5) - np.exp(-1.5))) < 1e-12
                   and abs(float(L.exact(1.0, 0.5, 1.5)[1]) - 0.1917) < 1e-4))
    # Euler gets better with smaller steps
    errs = []
    for dt in (0.5, 0.1, 0.01):
        te, y = L.euler(0.5, 1.5, dt, 8.0)
        errs.append(np.abs(y[:, 1] - L.exact(te, 0.5, 1.5)[1]).max())
    checks.append(("Euler error shrinks as dt shrinks", errs[0] > errs[1] > errs[2]))
    checks.append(("Euler with dt = 0.01 is within 0.005 mol/L", errs[2] < 0.005))
    te, y = L.euler(0.5, 1.5, 0.1, 5.0)
    checks.append(("Euler also conserves A + B + C = 1", bool(np.allclose(y.sum(axis=1), 1.0))))
    # edge case: k1 == k2 has its own formula; B = k t exp(-k t), peak at 1/k
    kk = 0.8
    checks.append(("edge: k1 = k2 peak time is 1/k and B is finite",
                   abs(L.peak_time(kk, kk) - 1 / kk) < 1e-12 and bool(np.isfinite(L.exact(1 / kk, kk, kk)[1]))))
    nearly = float(L.exact(1.0, 0.8, 0.8 + 1e-6)[1])
    checks.append(("edge: k2 very close to k1 agrees with the k1 = k2 formula", abs(nearly - float(L.exact(1.0, 0.8, 0.8)[1])) < 1e-4))
    # an experiment tells right from wrong
    times = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
    meas = L.pretend_experiment(0.5, 1.5, times)
    checks.append(("misfit is small for the true rate constants and large for wrong ones",
                   L.misfit(0.5, 1.5, times, meas) < 0.02 and L.misfit(1.0, 0.5, times, meas) > 0.05))
    # exercise answers used in the text: k1 = 0.4, k2 = 0.8 peaks at ln 2 / 0.4 = 1.733 s with B = 0.25
    tp2 = L.peak_time(0.4, 0.8)
    checks.append(("k1 = 0.4, k2 = 0.8: B peaks at 1.733 s with B = 0.25",
                   abs(tp2 - np.log(2) / 0.4) < 1e-12 and abs(float(L.exact(tp2, 0.4, 0.8)[1]) - 0.25) < 1e-12))
    # swapping k1 and k2 multiplies B by k2/k1 at every time, and the peak time stays the same
    tt = np.linspace(0.1, 6, 30)
    checks.append(("swapping k1 and k2 makes B exactly 3 times larger (k2/k1 = 3) with the same peak time",
                   bool(np.allclose(L.exact(tt, 1.5, 0.5)[1], 3 * L.exact(tt, 0.5, 1.5)[1])) and abs(L.peak_time(1.5, 0.5) - L.peak_time(0.5, 1.5)) < 1e-12))
    # at the peak of B: dB/dt = 0, i.e. k1 A = k2 B
    a_p, b_p, _ = L.exact(tp, 0.5, 1.5)
    checks.append(("at the peak of B the gain k1 A equals the loss k2 B", abs(0.5 * float(a_p) - 1.5 * float(b_p)) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
